# Renk Çubukları Özelleştirme

Bu notebook, PDS Handbook (TR) web sayfasının **Türkçe Jupyter karşılığıdır** — aynı açıklamalar, ders notları ve kod örnekleri.

| | |
|---|---|
| **Web sayfası** | `chapters/04-matplotlib/07-customizing-colorbars.html` |
| **Çalıştırma** | JupyterLab, VS Code veya Colab — hücreleri **yukarıdan aşağı** sırayla (`Shift+Enter`) |
| **Bağımlılık** | Kod hücreleri birbirine bağlıdır; hata alırsanız önce üsttekileri çalıştırın |

> **Kaynak:** Jake VanderPlas, *Python Data Science Handbook* — Türkçe ders uyarlaması



Orijinal: 04.07 Customizing Colorbars

Lejantlar ayrık noktaların ayrık etiketlerini gösterir.
    Nokta, çizgi veya bölgelerin rengine dayalı sürekli etiketler için etiketli bir renk çubuğu (colorbar) çok işe yarar.
    Matplotlib'de renk çubuğu, renklerin anlamına anahtar sağlayan ayrı bir eksen olarak çizilir.
    Kitap siyah-beyaz basıldığı için bu bölümün renkli şekilleri için
    çevrimiçi eki vardır.
    Çizim için not defterini hazırlayıp kullanacağımız işlevleri içe aktararak başlayalım:


In [ ]:
# import_plt_style.py
import matplotlib.pyplot as plt
plt.style.use('seaborn-white')



In [ ]:
# matplotlib_inline.py
%matplotlib inline
import numpy as np



Daha önce birkaç kez gördüğümüz gibi, en basit renk çubuğu plt.colorbar ile oluşturulur (aşağıdaki şekil):


In [ ]:
# colorbar_basic.py
x = np.linspace(0, 10, 1000)
I = np.sin(x) * np.cos(x[:, np.newaxis])

plt.imshow(I)
plt.colorbar();



> **Not**
>

Şimdi bu renk çubuklarını özelleştirme ve çeşitli durumlarda etkili kullanma fikirlerine bakalım.

## Renk çubuklarını özelleştirme

Renk haritası (colormap), görselleştirmeyi oluşturan çizim işlevinin cmap bağımsız değişkeniyle belirtilir (aşağıdaki şekil):


In [ ]:
# imshow_blues.py
plt.imshow(I, cmap='Blues');



Kullanılabilir renk haritalarının adları plt.cm ad alanındadır; IPython sekme tamamlama ile yerleşik listeyi görebilirsiniz:


```
plt.cm.<TAB>
```


Ancak renk haritası seçebilmek yalnızca ilk adımdır; asıl önemli olan olasılıklar arasında nasıl karar vereceğinizdir!
    Seçim ilk bakışta beklediğinizden çok daha incelikli olabilir.

### Renk haritası seçimi

Görselleştirmede renk seçiminin tam bir işlenmesi bu kitabın kapsamı dışındadır; konuyla ilgili eğlenceli okuma için Nicholas Rougier, Michael Droettboom ve Philip Bourne'un
    "Ten Simple Rules for Better Figures" makalesine bakın.
    Matplotlib çevrimiçi belgelerinde de renk haritası seçimi üzerine
    ayrıntılı bir tartışma vardır.

Genel olarak üç renk haritası kategorisini bilmelisiniz:

Matplotlib 2.0 öncesi varsayılan olan jet nitel bir haritaya örnektir.
    Varsayılan olması talihsizdi; nitel haritalar nicel veriyi göstermek için genelde kötü seçimdir.
    Sorunlardan biri, ölçek arttıkça parlaklıkta düzgün bir ilerleme olmamasıdır.

jet renk çubuğunu siyah-beyaza çevirerek bunu görebiliriz (aşağıdaki şekil):


In [ ]:
# grayscale_cmap.py
from matplotlib.colors import LinearSegmentedColormap

def grayscale_cmap(cmap):
    """Return a grayscale version of the given colormap"""
    cmap = plt.cm.get_cmap(cmap)
    colors = cmap(np.arange(cmap.N))
    
    # Convert RGBA to perceived grayscale luminance
    # cf. http://alienryderflex.com/hsp.html
    RGB_weight = [0.299, 0.587, 0.114]
    luminance = np.sqrt(np.dot(colors[:, :3] ** 2, RGB_weight))
    colors[:, :3] = luminance[:, np.newaxis]
        
    return LinearSegmentedColormap.from_list(
        cmap.name + "_gray", colors, cmap.N)
    

def view_colormap(cmap):
    """Plot a colormap with its grayscale equivalent"""
    cmap = plt.cm.get_cmap(cmap)
    colors = cmap(np.arange(cmap.N))
    
    cmap = grayscale_cmap(cmap)
    grayscale = cmap(np.arange(cmap.N))
    
    fig, ax = plt.subplots(2, figsize=(6, 2),
                           subplot_kw=dict(xticks=[], yticks=[]))
    ax[0].imshow([colors], extent=[0, 10, 0, 1])
    ax[1].imshow([grayscale], extent=[0, 10, 0, 1])



In [ ]:
# view_jet.py
view_colormap('jet')



### 🧪 Şimdi deneyin

🧪 Renk haritası karşılaştırın
      view_colormap('plasma') ve view_colormap('jet') yan yana düşünün; gri tonlamada hangisi daha düzgün?
          
      view_colormap('plasma')

Gri tonlamalı görüntüdeki parlak şeritlere dikkat edin.
    Tam renkte bile düzensiz parlaklık, gözün renk aralığının belirli bölümlerine çekilmesine ve veri setinin önemsiz kısımlarının vurgulanmasına yol açabilir.
    viridis (Matplotlib 2.0'dan beri varsayılan) gibi aralık boyunca eşit parlaklık değişimi için tasarlanmış bir harita daha iyidir; hem renk algısıyla uyumludur hem gri baskıya iyi aktarılır (aşağıdaki şekil):


In [ ]:
# view_viridis.py
view_colormap('viridis')



Ortalamadan pozitif/negatif sapmaları göstermek gibi durumlarda RdBu (*Kırmızı–Mavi*) gibi çift renkli çubuklar yardımcıdır. Ancak aşağıdaki şekilde görüldüğü gibi pozitif/negatif bilgi gri tonlamaya geçince kaybolabilir!


In [ ]:
# view_rdbu.py
view_colormap('RdBu')



Bu haritaların kullanımına devam ederken örnekler göreceğiz.

Matplotlib'de çok sayıda renk haritası vardır; listelemek için IPython ile plt.cm alt modülünü keşfedebilirsiniz. Python'da daha ilkeli bir renk yaklaşımı için Seaborn kütüphanesine bakın
    (Seaborn ile Görselleştirme).

### Renk sınırları ve uzatmalar

Matplotlib renk çubuğu özelleştirmesinde geniş olanak sunar.
    Renk çubuğu kendisi bir Axes örneğidir; gördüğümüz eksen ve işaret biçimlendirme yöntemleri geçerlidir.
    Örneğin renk sınırlarını daraltıp extend ile üst/alt sınır dışı değerleri üçgen okla gösterebiliriz.
    Gürültülü bir görüntü gösterirken işe yarayabilir (aşağıdaki şekil):


In [ ]:
# make noise in 1% of the image pixels
speckles = (np.random.random(I.shape) < 0.01)
I[speckles] = np.random.normal(0, 3, np.count_nonzero(speckles))

plt.figure(figsize=(10, 3.5))

plt.subplot(1, 2, 1)
plt.imshow(I, cmap='RdBu')
plt.colorbar()

plt.subplot(1, 2, 2)
plt.imshow(I, cmap='RdBu')
plt.colorbar(extend='both')
plt.clim(-1, 1)



Sol panelde varsayılan renk sınırları gürültülü piksellere uyum sağlar ve gürültü aralığı ilgilendiğimiz deseni tamamen bastırır.
    Sağ panelde sınırları elle ayarlayıp uzatmalar ekledik; sonuç veri için çok daha yararlı bir görselleştirmedir.

### Ayrık renk çubukları

Renk haritaları varsayılan olarak süreklidir; bazen ayrık değerleri temsil etmek istersiniz.
    Bunun en kolay yolu uygun bir harita adıyla birlikte istenen bölme sayısını plt.cm.get_cmap'e vermektir (aşağıdaki şekil):


In [ ]:
# discrete_colorbar.py
plt.imshow(I, cmap=plt.cm.get_cmap('Blues', 6))
plt.colorbar(extend='both')
plt.clim(-1, 1);



### 🧪 Şimdi deneyin

🧪 Ayrık renk çubuğu
      6 bölümlü bir Blues haritası ve extend='both' deneyin.
          
      plt.imshow(I, cmap=plt.cm.get_cmap('Blues', 6))
plt.colorbar(extend='both')
plt.clim(-1, 1)

Ayrık renk haritası diğerleri gibi kullanılabilir.

## Örnek: El yazısı rakamlar

Uygulama örneği: Scikit-Learn'deki rakamlar veri setinden el yazısı rakam görselleştirmesi; yaklaşık 2.000 adet $8 \times 8$ küçük resim.

Veri setini indirip birkaç örneği plt.imshow ile göstererek başlayalım (aşağıdaki şekil):


In [ ]:
# load images of the digits 0 through 5 and visualize several of them
from sklearn.datasets import load_digits
digits = load_digits(n_class=6)

fig, ax = plt.subplots(8, 8, figsize=(6, 6))
for i, axi in enumerate(ax.flat):
    axi.imshow(digits.images[i], cmap='binary')
    axi.set(xticks=[], yticks=[])



Her rakam 64 pikselin tonuyla tanımlandığından, her rakam 64 boyutlu uzayda bir nokta düşünülebilir: her boyut bir pikselin parlaklığıdır.
    Bu kadar yüksek boyutlu veriyi görselleştirmek zordur; bir yaklaşım, ilişkileri koruyarak boyutu düşüren boyut indirgeme (ör. manifold öğrenme) kullanmaktır.
    Boyut indirgeme denetimsiz makine öğrenmesine örnektir;
    Makine Öğrenmesi Nedir? bölümünde ayrıntılandırılacaktır.

Ayrıntıları erteleyerek rakam verisinin iki boyutlu bir manifold projeksiyonuna bakalım (ayrıntılar için
    Manifold Öğrenme):


In [ ]:
# project the digits into 2 dimensions using Isomap
from sklearn.manifold import Isomap
iso = Isomap(n_components=2, n_neighbors=15)
projection = iso.fit_transform(digits.data)



Sonuçları ayrık renk haritamızla göstereceğiz; estetik için ticks ve clim ayarlayacağız (aşağıdaki şekil):


In [ ]:
# plot the results
plt.scatter(projection[:, 0], projection[:, 1], lw=0.1,
            c=digits.target, cmap=plt.cm.get_cmap('plasma', 6))
plt.colorbar(ticks=range(6), label='digit value')
plt.clim(-0.5, 5.5)



Projeksiyon veri seti içindeki ilişkilere de ışık tutar: örneğin 2 ve 3 aralıkları neredeyse örtüşür; bazı 2 ve 3'lerin ayırt edilmesi zordur ve otomatik sınıflandırıcılar karıştırabilir.
    0 ve 1 gibi değerler daha uzaktır, karışma olasılığı daha düşüktür.

Manifold öğrenme ve rakam sınıflandırmasına
    Bölüm 5'te döneceğiz.

> **Not**
>
